# FASE 2. LIMPIEZA DE LOS DATOS - CALIDAD DEL AIRE (GVA)

## 1. Carga de los datos originales

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce

spark = SparkSession.builder \
    .appName("TFM_Limpieza_Aire_GVA") \
    .getOrCreate()

# Ruta de los archivos originales
ruta_base = "gs://tfm_unir1/aire/gva_quality/"

dataframes = []

for anio in range(2020, 2026):
    ruta = f"{ruta_base}gva_air_quality_{anio}.parquet"
    df_anio = (
        spark.read.parquet(ruta)
        .withColumn("noise", F.col("noise").cast("double")))
    dataframes.append(df_anio)

# Unir los seis años
df_gva_raw = reduce(
    lambda a, b: a.unionByName(b),
    dataframes)

# Comprobaciones iniciales
print("Registros cargados:", df_gva_raw.count())
print("Columnas:", len(df_gva_raw.columns))
print("Tipo de noise:", df_gva_raw.schema["noise"].dataType)

26/10/21 14:49:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/10/21 14:49:52 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/10/21 14:50:07 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


Registros cargados: 464088
Columnas: 45
Tipo de noise: DoubleType()


## 2. Selección de variables de interés

In [2]:
columnas_interes = [
    # Identificación y tiempo
    "station_code","station_name",
    "date","hour","datetime",

    # Contaminantes
    "no2","o3","pm10","pm2_5",

    # Meteorología
    "temperature","relative_humidity","wind_speed",
    "wind_direction_deg","precipitation","pressure","solar_radiation",

    # Ruido
    "noise",

    # Trazabilidad
    "stream","source_file"]

df_gva_work = df_gva_raw.select(*columnas_interes)

print("Registros:", df_gva_work.count())
print("Columnas seleccionadas:", len(df_gva_work.columns))

Registros: 464088
Columnas seleccionadas: 19


## 3. Estandarización de tipos de datos

### 3.1. Conversión de variables numéricas

In [3]:
from pyspark.sql import functions as F

# Variables que deben tener formato numérico decimal
columnas_numericas = [
    "no2","o3","pm10","pm2_5","temperature","relative_humidity","wind_speed",
    "wind_direction_deg","precipitation","pressure","solar_radiation","noise"]

# Crear un nuevo DataFrame estandarizado
df_gva_clean = df_gva_work

for columna in columnas_numericas:
    df_gva_clean = df_gva_clean.withColumn(columna,
        F.col(columna).cast("double"))

# Comprobar los tipos de datos resultantes
df_gva_clean.select(*columnas_numericas).printSchema()

root
 |-- no2: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- relative_humidity: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction_deg: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- solar_radiation: double (nullable = true)
 |-- noise: double (nullable = true)



## 4. Tratamiento de valores inválidos y ausentes

### 4.1. Limpieza de concentraciones atmosféricas

In [4]:
from pyspark.sql import functions as F

contaminantes = ["no2", "o3", "pm10", "pm2_5"]

# Crear columnas limpias sin modificar las originales
for c in contaminantes:
    df_gva_clean = df_gva_clean.withColumn(
        f"{c}_clean",
        F.when(
            (F.col(c) >= 0) & (~F.isnan(F.col(c))),F.col(c)
        ).otherwise(None))

# Comprobar el resultado
df_gva_clean.select(
    "no2", "no2_clean",
    "o3", "o3_clean",
    "pm10", "pm10_clean",
    "pm2_5", "pm2_5_clean"
).show(5, truncate=False)

+----+---------+----+--------+----+----------+-----+-----------+
|no2 |no2_clean|o3  |o3_clean|pm10|pm10_clean|pm2_5|pm2_5_clean|
+----+---------+----+--------+----+----------+-----+-----------+
|18.0|18.0     |null|null    |21.0|21.0      |16.0 |16.0       |
|18.0|18.0     |null|null    |20.0|20.0      |16.0 |16.0       |
|22.0|22.0     |null|null    |24.0|24.0      |18.0 |18.0       |
|20.0|20.0     |null|null    |24.0|24.0      |19.0 |19.0       |
|19.0|19.0     |null|null    |24.0|24.0      |19.0 |19.0       |
+----+---------+----+--------+----+----------+-----+-----------+
only showing top 5 rows



### 4.2. Tratamiento de valores inválidos en variables meteorológicas

In [5]:
from pyspark.sql import functions as F

# Variables meteorológicas
variables_meteo = [
    "temperature","relative_humidity","wind_speed","wind_direction_deg",
    "precipitation","pressure","solar_radiation"]

# Crear columnas limpias conservando las originales
for c in variables_meteo:
    df_gva_clean = df_gva_clean.withColumn(
        f"{c}_clean",
        F.when(
            F.col(c).isNotNull() &
            (~F.isnan(F.col(c))),
            F.col(c)).otherwise(None))

# Comprobar las columnas creadas
df_gva_clean.select(
    "station_code",
    "temperature_clean",
    "relative_humidity_clean",
    "wind_speed_clean",
    "precipitation_clean"
).show(10, truncate=False)

+------------+-----------------+-----------------------+----------------+-------------------+
|station_code|temperature_clean|relative_humidity_clean|wind_speed_clean|precipitation_clean|
+------------+-----------------+-----------------------+----------------+-------------------+
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null            |null               |
|46250030    |null             |null                   |null

## 5. Estandarización de variables temporales

### 5.1. Creación de variables de calendario

In [6]:
from pyspark.sql import functions as F

df_gva_clean = (
    df_gva_clean
    .withColumn("anio", F.year("datetime"))
    .withColumn("mes", F.month("datetime"))
    .withColumn("dia", F.dayofmonth("datetime"))
    .withColumn("hora_dia", F.hour("datetime"))
    .withColumn("dia_semana", F.dayofweek("datetime")))

# Comprobar el resultado
df_gva_clean.select(
    "station_code","datetime","anio",
    "mes","dia","hora_dia","dia_semana"
).show(10, truncate=False)

+------------+-------------------+----+---+---+--------+----------+
|station_code|datetime           |anio|mes|dia|hora_dia|dia_semana|
+------------+-------------------+----+---+---+--------+----------+
|46250030    |2020-08-01 00:00:00|2020|8  |1  |0       |7         |
|46250030    |2020-08-01 01:00:00|2020|8  |1  |1       |7         |
|46250030    |2020-08-01 02:00:00|2020|8  |1  |2       |7         |
|46250030    |2020-08-01 03:00:00|2020|8  |1  |3       |7         |
|46250030    |2020-08-01 04:00:00|2020|8  |1  |4       |7         |
|46250030    |2020-08-01 05:00:00|2020|8  |1  |5       |7         |
|46250030    |2020-08-01 06:00:00|2020|8  |1  |6       |7         |
|46250030    |2020-08-01 07:00:00|2020|8  |1  |7       |7         |
|46250030    |2020-08-01 08:00:00|2020|8  |1  |8       |7         |
|46250030    |2020-08-01 09:00:00|2020|8  |1  |9       |7         |
+------------+-------------------+----+---+---+--------+----------+
only showing top 10 rows



## 6. Preparación de las mediciones de ruido

### 6.1. Tratamiento de valores ausentes y no utilizables

In [7]:
from pyspark.sql import functions as F

# Crear columna de ruido preparada
df_gva_clean = df_gva_clean.withColumn(
    "noise_clean",
    F.when(
        F.col("noise").isNotNull() &
        (~F.isnan(F.col("noise"))) &
        (F.col("noise") >= 0),
        F.col("noise")).otherwise(None))

# Comprobar la conservación de las mediciones
df_gva_clean.agg(
    F.count("noise").alias("ruido_original_no_nulo"),
    F.count("noise_clean").alias("ruido_utilizable"),
    F.sum(
        F.when(F.col("noise_clean").isNull(), 1).otherwise(0)
    ).alias("ruido_no_disponible")).show()

+----------------------+----------------+-------------------+
|ruido_original_no_nulo|ruido_utilizable|ruido_no_disponible|
+----------------------+----------------+-------------------+
|                 23648|           23648|             440440|
+----------------------+----------------+-------------------+



## 7. Validación final de los datos preparados

### 7.1. Comprobación de integridad

In [8]:
# Comparar los registros originales y preparados

registros_originales = df_gva_raw.count()
registros_preparados = df_gva_clean.count()

print("Registros originales:", registros_originales)
print("Registros preparados:", registros_preparados)
print("Diferencia:", registros_originales - registros_preparados)

if registros_originales == registros_preparados:
    print("OK: No se han perdido registros")
else:
    print("REVISAR: Existen diferencias")

Registros originales: 464088
Registros preparados: 464088
Diferencia: 0
OK: No se han perdido registros


 ## 8. Almacenamiento de los datos preparados

### 8.1. Exportación a formato Parquet

In [9]:
# Ruta de salida
ruta_gva_limpio = "gs://tfm_unir1/processed/aire/gva/"

# Guardar los datos preparados
df_gva_clean.write \
    .mode("errorifexists") \
    .parquet(ruta_gva_limpio)

print("Datos GVA preparados y guardados correctamente")

26/10/21 15:00:54 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Datos GVA preparados y guardados correctamente
